# 04 — Modeling

Xây dựng và đánh giá mô hình dự đoán:
- Phân lớp phân khúc khách hàng (LogReg / DT / RF)
- Dự báo doanh số theo chuỗi thời gian (Naive / MA / ARIMA / Holt-Winters / Prophet)

In [ ]:
# %% [import] Thư viện và cấu hình
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.data import load_params, load_processed_data
from src.features import build_monthly_sales
from src.models import run_supervised_pipeline, run_forecasting_pipeline
from src.models.supervised import error_analysis, get_feature_importance
from src.evaluation import (
    save_table, get_confusion_matrix,
    summarize_classification, summarize_forecasting,
)
from src.visualization import (
    plot_confusion_matrix, plot_feature_importance,
    plot_forecast, plot_residuals,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
# %% [load] Đọc dữ liệu đã tiền xử lý
params = load_params()
df     = load_processed_data(params)
print(f"Shape: {df.shape}")

In [ ]:
# %% [config] Ghi rõ hyperparams và thiết lập thực nghiệm
import time

print("=== THIẾT LẬP THỰC NGHIỆM ===")
print(f"Seed             : {params['general']['seed']}")
print(f"Test size        : {params['preprocessing']['test_size']}")
print(f"CV folds         : {params['models']['supervised']['cv_folds']}")
print(f"\n--- Random Forest ---")
print(f"  n_estimators   : {params['models']['supervised']['random_forest']['n_estimators']}")
print(f"  max_depth      : {params['models']['supervised']['random_forest']['max_depth']}")
print(f"\n--- Decision Tree ---")
print(f"  max_depth      : {params['models']['supervised']['decision_tree']['max_depth']}")
print(f"\n--- Logistic Regression ---")
print(f"  max_iter       : {params['models']['supervised']['logistic_regression']['max_iter']}")
print(f"\n--- Forecasting ---")
print(f"  ARIMA order    : {params['models']['forecasting']['arima']['order']}")
print(f"  Forecast periods: {params['models']['forecasting']['forecast_periods']}")

## PHẦN A — Phân lớp phân khúc khách hàng (Classification)

### 1. Chuẩn bị dữ liệu

In [ ]:
# %% [prepare] Tách features và target, chia train/test stratified
from src.models.supervised import prepare_xy, split_data, build_models

X, y, le = prepare_xy(df, params)
X_train, X_test, y_train, y_test = split_data(X, y, params)

print(f"Classes    : {list(le.classes_)}")
print(f"X_train    : {X_train.shape} | X_test: {X_test.shape}")
print(f"\nPhân phối nhãn train:")
print(pd.Series(y_train).value_counts())

In [ ]:
# %% [plot] Phân phối Segment trong dataset
fig, ax = plt.subplots(figsize=(7, 4))
seg_counts = df[params["preprocessing"]["target_column"]].value_counts()
sns.barplot(x=seg_counts.index, y=seg_counts.values, ax=ax, palette="Blues_r")
ax.set_title("Phân phối Segment (target)")
ax.set_ylabel("Số lượng")
plt.tight_layout()
plt.savefig("../outputs/figures/segment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### 2. Train và so sánh models

In [ ]:
# %% [train] Train toàn bộ pipeline phân lớp: LogReg, DT, RF — đo thời gian train
start = time.time()

models, clf_results, le, (X_train, X_test, y_train, y_test) = \
    run_supervised_pipeline(df, params)

print(f"\nTổng thời gian train: {time.time() - start:.2f}s")

clf_summary = summarize_classification(clf_results.to_dict("records"))
save_table(clf_summary, "classification_results.csv", params)
clf_summary

In [ ]:
# %% [plot] Barplot so sánh F1-macro và ROC-AUC các models
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.barplot(data=clf_summary, x="model", y="f1_macro",  ax=axes[0], palette="Blues_r")
axes[0].set_title("F1-macro theo model")
axes[0].set_ylim(0, 1)

sns.barplot(data=clf_summary, x="model", y="roc_auc", ax=axes[1], palette="Oranges_r")
axes[1].set_title("ROC-AUC theo model")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("../outputs/figures/classification_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### 3. Phân tích lỗi — Random Forest (model tốt nhất)

In [ ]:
# %% [error] Confusion matrix và classification report của Random Forest
error_analysis(models["random_forest"], X_test, y_test, le)

In [ ]:
# %% [plot] Vẽ confusion matrix dạng heatmap
cm = get_confusion_matrix(
    y_test,
    models["random_forest"].predict(X_test),
    list(le.classes_),
)
plot_confusion_matrix(cm, params)
plt.show()

In [ ]:
# %% [importance] Feature importance của Random Forest — top 15
fi = get_feature_importance(models["random_forest"], list(X_train.columns))
plot_feature_importance(fi, params, top_n=15)
save_table(fi, "feature_importance.csv", params)
plt.show()

## PHẦN B — Dự báo doanh số chuỗi thời gian (Forecasting)

### 4. Chuẩn bị chuỗi thời gian

In [ ]:
# %% [prepare] Tổng hợp doanh số theo tháng, kiểm định tính dừng ADF
from src.models.forecasting import adf_test, time_split

monthly = build_monthly_sales(df)
n_test  = params["models"]["forecasting"]["forecast_periods"]
train, test = time_split(monthly, n_test)

adf_result = adf_test(train["Sales"])
print(f"ADF Statistic : {adf_result['adf_statistic']:.4f}")
print(f"p-value       : {adf_result['p_value']:.4f}")
print(f"Stationary    : {adf_result['is_stationary']}")
print(f"\nTrain: {len(train)} months | Test: {n_test} months")
print(f"Train: {train['YearMonth'].iloc[0].date()} → {train['YearMonth'].iloc[-1].date()}")
print(f"Test : {test['YearMonth'].iloc[0].date()} → {test['YearMonth'].iloc[-1].date()}")

In [ ]:
# %% [plot] Visualize train/test split chuỗi thời gian
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train["YearMonth"], train["Sales"], label="Train", color="steelblue", linewidth=2)
ax.plot(test["YearMonth"],  test["Sales"],  label="Test",  color="black", linewidth=2)
ax.axvline(x=test["YearMonth"].iloc[0], color="red", linestyle="--", label="Split point")
ax.set_title("Train / Test Split theo thời gian")
ax.set_xlabel("Tháng")
ax.set_ylabel("Sales (USD)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/figures/train_test_split.png", dpi=150, bbox_inches="tight")
plt.show()

### 5. Chạy và so sánh các models

In [ ]:
# %% [forecast] Chạy toàn bộ pipeline forecasting: Naive, MA, ARIMA, Holt-Winters, Prophet
fc_results, forecasts, train, test, residual = \
    run_forecasting_pipeline(monthly, params)

fc_summary = summarize_forecasting(fc_results)
save_table(fc_summary, "forecasting_results.csv", params)
fc_summary

In [ ]:
# %% [plot] Forecast vs Actual — tất cả models
plot_forecast(train, test, forecasts, params)
plt.show()

In [ ]:
# %% [plot] Barplot so sánh MAE, RMSE, sMAPE các models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, color in zip(axes,
                              ["MAE", "RMSE", "sMAPE"],
                              ["steelblue", "coral", "green"]):
    sns.barplot(data=fc_summary, x="model", y=metric, ax=ax, color=color)
    ax.set_title(f"{metric} theo model")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("../outputs/figures/forecasting_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Phân tích residual — model tốt nhất

In [ ]:
# %% [residual] Phân tích residual: mean, std, kiểm định ADF
best_model = fc_summary.iloc[0]["model"]
print(f"Best model    : {best_model}")
print(f"Residual mean : {residual['mean']:.4f}")
print(f"Residual std  : {residual['std']:.4f}")
print(f"ADF p-value   : {residual['adf']['p_value']:.4f}")
print(f"Stationary    : {residual['adf']['is_stationary']}")

In [ ]:
# %% [plot] Residual plot theo thời gian và phân phối
plot_residuals(residual["residuals"], params)
plt.show()